# ForgeLM — DPO Preference Alignment

Align a language model with human preferences using Direct Preference Optimization.

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/dpo_alignment.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
# Pinned to v0.5.7; bump on each release
!pip install -q --no-cache-dir 'forgelm[qlora]==0.5.7' bitsandbytes
!pip uninstall -y wandb -q
# Colab base image ships torchao 0.10; modern peft requires >=0.16 for PeftModel.from_pretrained.
# Upgrade so the comparison cell at the end can load the trained LoRA adapter.
!pip install -q --upgrade 'torchao>=0.16.0'
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create DPO config
# DPO requires a dataset with 'chosen' and 'rejected' columns
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "dpo"
  dpo_beta: 0.1
  output_dir: "./dpo_checkpoints"
  max_steps: 100
  per_device_train_batch_size: 4
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "argilla/ultrafeedback-binarized-preferences"
"""

with open("dpo_config.yaml", "w") as f:
    f.write(config_yaml)
print("DPO config saved! (max_steps: 100)")

In [ ]:
# Step 4: Validate
!forgelm --config dpo_config.yaml --dry-run

In [ ]:
# Step 5: Train
!forgelm --config dpo_config.yaml

In [ ]:
# Step 6: Verify output
import os

model_path = "./dpo_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"ERROR: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"ERROR: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 7: Compare base model vs DPO-aligned model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./dpo_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: DPO model not found. Ensure training completed successfully.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading DPO-aligned model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # DPO should improve helpfulness and reduce harmful/unhelpful patterns
    test_prompts = [
        "Explain quantum computing in simple terms.",
        "What are the benefits of exercise?",
        "How should I prepare for a job interview?",
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[DPO-ALIGNED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("DPO-aligned model should produce more helpful, preferred responses.")

## Step 8: Download or push your trained adapter

The training cell above wrote a PEFT adapter (`adapter_config.json` +
`adapter_model.safetensors` + tokenizer files, ~30-100 MB total). Two
ways to get it off Colab:

- **Local download** — ZIP the adapter and trigger a browser download.
  The ZIP lands on your filesystem; extract anywhere and load back via
  `PeftModel.from_pretrained(base, "./extracted_folder")`.
- **HuggingFace Hub push** — push directly to a repo for sharing /
  versioning. Requires `HF_TOKEN` in Colab Secrets with write access to
  the target repo. Uncomment the **Option B** block.

If your Colab session is about to time out, run this cell first — the
adapter doesn't survive a runtime disconnect.

In [ ]:
# Option A — ZIP + Colab browser download (default)
import os
import shutil
from pathlib import Path

# Reuse the adapter directory declared in the compare cell above so the
# path is single-sourced (Sonar S1192). Run this cell after Step 6+7.
MODEL_PATH = model_path  # noqa: F821 — set in the compare cell above
ZIP_NAME = "smollm2_dpo_finetune"

if not Path(MODEL_PATH).is_dir():
    print(f"No trained adapter at {MODEL_PATH}. Run the training cell above first.")
else:
    archive = shutil.make_archive(ZIP_NAME, "zip", root_dir=MODEL_PATH)
    archive_size = Path(archive).stat().st_size
    print(f"Created {archive} ({archive_size:,} bytes)")
    try:
        from google.colab import files  # type: ignore

        files.download(archive)
        print("Browser download triggered.")
    except ImportError:
        print(f"Local Jupyter — grab the ZIP from {Path(archive).resolve()}")

# ---------------------------------------------------------------------------
# Option B — Push to HuggingFace Hub (uncomment + edit REPO_ID to use)
# ---------------------------------------------------------------------------
# Set HF_TOKEN in Colab → 🔑 Secrets first (with write access to your repo).
#
# from huggingface_hub import HfApi
# try:
#     from google.colab import userdata  # type: ignore
# except ImportError:
#     userdata = None  # local Jupyter — assume HF_TOKEN is already in the env
# if userdata is not None:
#     # If the Colab Secret is missing, ``userdata.get`` raises a clear
#     # SecretNotFoundError — let it surface rather than silently swallowing.
#     os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
#
# REPO_ID = "your-username/smollm2_dpo_finetune"  # change to your HF Hub repo
# api = HfApi()
# api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)
# api.upload_folder(folder_path=MODEL_PATH, repo_id=REPO_ID, repo_type="model")
# print(f"Pushed to https://huggingface.co/{REPO_ID}")

## Other Alignment Methods

Change `trainer_type` in your config to switch methods:

| Method | `trainer_type` | Dataset Format | Best For |
|--------|---------------|----------------|----------|
| DPO | `"dpo"` | chosen/rejected pairs | General preference alignment |
| SimPO | `"simpo"` | chosen/rejected pairs | Lower memory (no reference model) |
| KTO | `"kto"` | completion + label (bool) | Binary feedback (production data) |
| ORPO | `"orpo"` | chosen/rejected pairs | SFT + alignment in one stage |
| GRPO | `"grpo"` | prompt only | Reasoning RL (like DeepSeek-R1) |